In [1]:
import warnings 
import pandas as pd
import seaborn as sns

from glob import glob 

warnings.filterwarnings('ignore')
%matplotlib inline
%load_ext autoreload
%autoreload 2

pd.options.display.float_format = "{:,.3f}".format
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

INFILE_META = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/meta.csv'
INFILE_SDTO = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/Sample_Donor_Tissue_Origin_01_June_2020.csv'
INFILE_COMBI = '/home/grace/work/PPCG_DifferentialGenesetMutation/data/Combimets_Sample_Metadata_20230911.grace.csv'
DATA_BASEDIR = '/home/grace/work/PPCG_DifferentialGenesetMutation/data'
OUTFILE_SAMPLESHEET = '/home/grace/work/PPCG_DifferentialGenesetMutation/samplesheet.angel.alldonors.tsv'


In [2]:
meta = pd.read_csv(INFILE_META, sep=',', header=0)
meta['sample'] = meta['id'].apply(lambda x: x[:9])
wgd_LUT = meta.set_index('sample')['is.wgd'].to_dict()
ploidy_LUT = meta.set_index('sample')['mploid'].to_dict()
purity_LUT = meta.set_index('sample')['purity'].to_dict()
meta.head()

,id,is.wgd,mploid,purity,sample
0,PPCG0001a_DNA_vs_PPCG0001b_DNA,False,1.886,0.280,PPCG0001a
1,PPCG0002a_DNA_vs_PPCG0002b_DNA,False,2.311,0.498,PPCG0002a
2,PPCG0003a_DNA_vs_PPCG0003b_DNA,False,2.385,0.778,PPCG0003a
3,PPCG0004a_DNA_vs_PPCG0004b_DNA,False,1.689,0.478,PPCG0004a
4,PPCG0005a_DNA_vs_PPCG0005b_DNA,False,2.000,0.673,PPCG0005a


In [3]:
sdto = pd.read_csv(INFILE_SDTO, sep=',', header=0)
sdto['sample'] = sdto['PPCG_Sample_ID'].apply(lambda x: x[:9])
tissue_LUT = sdto.set_index('sample')['Tissue_Origin'].to_dict()
sdto.head()

,Country,PPCG_Donor_ID,PPCG_Sample_ID,Tissue_Origin,sample
0,Australia,PPCG0126,PPCG0126a_DNA,Primary,PPCG0126a
1,Australia,PPCG0127,PPCG0127a_DNA,Primary,PPCG0127a
2,Australia,PPCG0128,PPCG0128a_DNA,Primary,PPCG0128a
3,Australia,PPCG0129,PPCG0129a_DNA,Primary,PPCG0129a
4,Australia,PPCG0130,PPCG0130a_DNA,Primary,PPCG0130a


In [4]:
combi = pd.read_csv(INFILE_COMBI, sep=',', header=0)
combi['sample'] = combi['WGS_AssayID'].apply(lambda x: x[:9])
combi.head()

,Country,Centre,Local_Donor_ID,PPCG_Donor_ID,WGS_AssayID,First_Age,Procedure_Date,Days_Offset,Local_Sample_ID,Gundem_ID,Hong_ID,Matched_Normal_PPCG_Assay_ID,Matched_Normal_Local_Sample_ID,tube_id,treated,procedure_date,tumour_sample_timing,tumour_clinical_timinig,tumour_type,sample_type,tumour_region,metastasis_extent,sample
0,Australia,Melbourne,CMHP1,PPCG0387,PPCG0387a_DNA,64,2008-05-19,0,CMHS1,NaN,299.000,PPCG0387b_DNA,CMHS5,299bT,0,19/5/2008,longitudinal,synchronous,Primary,primary,prostate,NaN,PPCG0387a
1,Australia,Melbourne,CMHP1,PPCG0387,PPCG0387c_DNA,64,2011-04-29,1075,CMHS2,NaN,299.000,PPCG0387b_DNA,CMHS5,299aM,1,29/4/2011,longitudinal,metachronous,Left Shoulder Bone Metastasis,bone,bone,distant,PPCG0387c
2,Australia,Melbourne,CMHP2,PPCG0388,PPCG0388e_DNA,71,2011-06-02,1243,CMHS10,NaN,498.000,PPCG0388b_DNA,CMHS15,498aR,0,2/6/2011,longitudinal,metachronous,Recurrence,recurrence,visceral,distant,PPCG0388e
3,Australia,Melbourne,CMHP2,PPCG0388,PPCG0388a_DNA,71,2008-01-06,0,CMHS6,NaN,498.000,PPCG0388b_DNA,CMHS15,498bT,0,6/1/2008,longitudinal,synchronous,Primary,primary,prostate,NaN,PPCG0388a
4,Australia,Melbourne,CMHP2,PPCG0388,PPCG0388c_DNA,71,2011-06-03,1244,CMHS7,NaN,498.000,PPCG0388b_DNA,CMHS15,498aM,0,3/6/2011,longitudinal,metachronous,Right Sacrum Metastasis Hormone Naïve,bone,bone,distant,PPCG0388c


In [5]:
cna_samples = set([x.split('/')[-1][:9] for x in glob(f"{DATA_BASEDIR}/scna/*.txt")])
indel_samples = set([x.split('/')[-1][:9] for x in glob(f"{DATA_BASEDIR}/indel/*.vcf")])
snv_samples = set([x.split('/')[-1][:9] for x in glob(f"{DATA_BASEDIR}/snv/*.vcf")])
sv_samples = set([x.split('/')[-1][:9] for x in glob(f"{DATA_BASEDIR}/sv_annot/*.vcf")])

meta_samples = set(meta['sample'].unique())
sheet_samples = set(sdto['sample'].unique())
combi_samples = set(combi['sample'].unique())

all_samples = set()
all_samples = all_samples | meta_samples
all_samples = all_samples | sheet_samples
all_samples = all_samples | combi_samples
all_samples = all_samples | cna_samples
all_samples = all_samples | indel_samples
all_samples = all_samples | snv_samples
all_samples = all_samples | sv_samples

print(f"meta: {len(meta_samples)}")
print(f"sheet: {len(sheet_samples)}")
print(f"combi: {len(combi_samples)}")
print(f"indel: {len(indel_samples)}")
print(f"cna: {len(cna_samples)}")
print(f"snv: {len(snv_samples)}")
print(f"sv: {len(sv_samples)}")
print()
print(len(all_samples))

meta: 1214
sheet: 1202
combi: 225
indel: 1147
cna: 1202
snv: 1209
sv: 1089

1220


In [6]:
combi_LUT = {s: s in combi_samples for s in all_samples}
combi_LUT['PPCG1107a'] = True

sheet = pd.DataFrame(index=sorted(all_samples))
sheet['tissue'] = tissue_LUT
sheet['purity'] = purity_LUT
sheet['ploidy'] = ploidy_LUT
sheet['WGD'] = wgd_LUT
sheet['CombiMets'] = combi_LUT
sheet['cohort'] = sheet['CombiMets'].map({True: 'COMBI', False: 'PPCG'})
sheet = sheet.drop('CombiMets', axis=1)
sheet = sheet.reset_index()
sheet = sheet.rename(columns={'index': 'sample'})
sheet['donor'] = sheet['sample'].apply(lambda x: x[:8])
sheet = sheet[['donor', 'sample', 'tissue', 'purity', 'ploidy', 'WGD', 'cohort']].copy()

cnafiles = glob(f"{DATA_BASEDIR}/scna/*.txt")
snvfiles = glob(f"{DATA_BASEDIR}/snv/*.vcf")
indelfiles = glob(f"{DATA_BASEDIR}/indel/*.vcf")
svfiles = glob(f"{DATA_BASEDIR}/sv_annot/*.vcf")

labels = ['CNA', 'SNV', 'INDEL', 'SV']
dirlist = [cnafiles, snvfiles, indelfiles, svfiles]

for label, dirfiles in zip(labels, dirlist):
    present = set([x.split('/')[-1][:9] for x in dirfiles])
    sheet[label] = sheet['sample'].apply(lambda x: x in present)

sheet.head()

,donor,sample,tissue,purity,ploidy,WGD,cohort,CNA,SNV,INDEL,SV
0,PPCG0001,PPCG0001a,Primary,0.280,1.886,False,PPCG,True,True,True,True
1,PPCG0002,PPCG0002a,Primary,0.498,2.311,False,PPCG,True,True,True,True
2,PPCG0003,PPCG0003a,Primary,0.778,2.385,False,PPCG,True,True,True,True
3,PPCG0004,PPCG0004a,Primary,0.478,1.689,False,PPCG,True,True,True,True
4,PPCG0005,PPCG0005a,Primary,0.673,2.000,False,PPCG,True,True,True,False


In [7]:
# update purity & ploidy using new Angel data 
pp = None 
filepaths = glob(f"{DATA_BASEDIR}/angel/battenberg_scna/Cellularity_Ploidy/*.txt")
for path in filepaths:
    sample = path.split('/')[-1][:9]
    df = pd.read_csv(path, sep='\t', header=0)
    df['sample'] = sample
    if pp is None:
        pp = df 
    else:
        assert set(pp.columns.to_list()) == set(df.columns.to_list())
        pp = pd.concat([pp, df], ignore_index=True)

sheet = sheet.set_index('sample')
for rec in pp.itertuples():
    sheet.loc[rec.sample, 'purity'] = rec.cellularity
    sheet.loc[rec.sample, 'ploidy'] = rec.ploidy
sheet = sheet.reset_index()
sheet = sheet.copy()
sheet.head()


,sample,donor,tissue,purity,ploidy,WGD,cohort,CNA,SNV,INDEL,SV
0,PPCG0001a,PPCG0001,Primary,0.280,1.886,False,PPCG,True,True,True,True
1,PPCG0002a,PPCG0002,Primary,0.498,2.311,False,PPCG,True,True,True,True
2,PPCG0003a,PPCG0003,Primary,0.778,2.385,False,PPCG,True,True,True,True
3,PPCG0004a,PPCG0004,Primary,0.478,1.689,False,PPCG,True,True,True,True
4,PPCG0005a,PPCG0005,Primary,0.673,2.000,False,PPCG,True,True,True,False


In [8]:
# FILTERING
print(sheet['donor'].nunique())

# remove donors without required information. 
sheet['tissue'] = sheet['tissue'].fillna('Unknown')
sheet = sheet.dropna(subset=['purity', 'ploidy', 'WGD'])
# sheet[sheet[['purity', 'ploidy', 'WGD']].isna().any(axis=1)]
print(sheet['donor'].nunique())

# remove donors without any variant data
sheet['data'] = sheet[labels].any(axis=1)
sheet = sheet[sheet['data']==True]
sheet = sheet.drop('data', axis=1)
sheet['Missing_Files'] = ~sheet[['CNA', 'SNV', 'INDEL', 'SV']].all(axis=1)
sheet['Missing_Files'] = sheet['Missing_Files'].map({True: 'Yes', False: 'No'}).astype(str)
print(sheet['donor'].nunique())

# not weird source 
whitelist = ['Primary','Metastasis','Recurrence']
sheet = sheet[sheet['tissue'].isin(whitelist)]

# # has at least 1 data source 
# sheet['data'] = sheet[labels].any(axis=1)
# sheet = sheet[sheet['data']==True]
# sheet = sheet.drop('data', axis=1)
# sheet['Missing_Data'] = ~sheet[['CNA', 'SNV', 'INDEL', 'SV']].all(axis=1)

# # purity >= 0.25 if PPCG
# sheet = sheet[(sheet['CombiMets']==True) | (sheet['purity']>=0.25)]

# # missing some variant data if PPCG
# sheet = sheet[(sheet['CombiMets']==True) | (sheet['Missing_Data']==False)]

# print()
# print(sheet.groupby('CombiMets')['CNA'].value_counts())
# print()
# print(sheet.groupby('CombiMets')['SNV'].value_counts())
# print()
# print(sheet.groupby('CombiMets')['INDEL'].value_counts())
# print()
# print(sheet.groupby('CombiMets')['SV'].value_counts())

print(sheet.groupby('cohort')['tissue'].value_counts())
print()

sheet = sheet.sort_values('purity')
print()
print(sheet[sheet['cohort']=='COMBI'].head())
print()
print(sheet[sheet['cohort']=='PPCG'].head())


1006
1004
1004
cohort  tissue    
COMBI   Metastasis    145
        Primary        73
        Recurrence      2
PPCG    Primary       933
Name: count, dtype: int64


         sample     donor      tissue  purity  ploidy    WGD cohort   CNA   SNV  INDEL    SV Missing_Files
1180  PPCG1078c  PPCG1078  Metastasis   0.118   1.962  False  COMBI  True  True   True  True            No
1134  PPCG1032c  PPCG1032  Metastasis   0.120   1.987  False  COMBI  True  True   True  True            No
497   PPCG0388d  PPCG0388  Metastasis   0.171   3.752   True  COMBI  True  True   True  True            No
1128  PPCG1026c  PPCG1026  Metastasis   0.173   2.065  False  COMBI  True  True   True  True            No
551   PPCG0429a  PPCG0429  Metastasis   0.176   1.930  False  COMBI  True  True   True  True            No

        sample     donor   tissue  purity  ploidy    WGD cohort   CNA   SNV  INDEL     SV Missing_Files
277  PPCG0194a  PPCG0194  Primary   0.133   2.028  False   PPCG  True  True   True   Tr

In [9]:
# FORMAT & WRITE TO FILE
def stringify_missing_data(row: pd.Series) -> str:
    missing = [] 
    for field in ['CNA', 'SNV', 'INDEL', 'SV']:
        if row[field] == False:
            missing.append(field)
    if len(missing) == 0:
        return '.'
    return ','.join(sorted(missing))

print(sheet.groupby('cohort')['Missing_Files'].value_counts())
print()
sheet['Missing_Files'] = sheet.apply(stringify_missing_data, axis=1)
sheet = sheet.drop(['CNA', 'SNV', 'INDEL', 'SV'], axis=1)
sheet = sheet.sort_values(['cohort', 'sample'])
sheet.to_csv(OUTFILE_SAMPLESHEET, sep='\t', index=False, float_format='%.3f')

print(sheet['tissue'].value_counts())
sheet[sheet['Missing_Files']!='.'].head(10)
# sheet.head(10)

cohort  Missing_Files
COMBI   No               204
        Yes               16
PPCG    No               864
        Yes               69
Name: count, dtype: int64

tissue
Primary       1006
Metastasis     145
Recurrence       2
Name: count, dtype: int64


,sample,donor,tissue,purity,ploidy,WGD,cohort,Missing_Files
246,PPCG0179j,PPCG0179,Metastasis,0.804,2.838,True,COMBI,INDEL
510,PPCG0395c,PPCG0395,Metastasis,0.350,2.018,False,COMBI,"INDEL,SV"
520,PPCG0403a,PPCG0403,Metastasis,0.950,2.001,False,COMBI,SV
540,PPCG0424a,PPCG0424,Primary,0.340,2.000,False,COMBI,SV
541,PPCG0424c,PPCG0424,Recurrence,0.360,2.015,False,COMBI,SV
542,PPCG0425a,PPCG0425,Metastasis,0.320,2.000,False,COMBI,SV
545,PPCG0427a,PPCG0427,Primary,0.250,2.000,False,COMBI,"INDEL,SV"
549,PPCG0428c,PPCG0428,Metastasis,0.310,4.000,True,COMBI,"INDEL,SV"
550,PPCG0428d,PPCG0428,Metastasis,0.320,2.000,False,COMBI,"INDEL,SV"
552,PPCG0430a,PPCG0430,Metastasis,0.950,1.957,False,COMBI,"INDEL,SV"
